# 定位结果数据分析与可视化

**数据输入**：
- `locations_{SOL_METHOD}_{DATA_SET}_mm_DD_HHMMSS.csv`：定位计算结果（支持多算法对比）
- `ground_truths_{DATA_SET}.csv`：地面实测基准数据

**功能特性**：
- 支持同时对比 2~5 组算法的精度表现
- 支持按数据集（Dataset）筛选后进行针对性分析

## 第1部分：初始化与配置


In [ ]:
# ====== 导入依赖库 ======
import os
from os.path import join, abspath
from glob import glob
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import gaussian_kde
from datetime import date
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scienceplots

# ====== 手动配置参数 ======

# ---- 待比较的算法列表（格式: 名称, 颜色）----
# 支持同时对比 2~5 组算法；颜色用于图表区分 '#00FFFF'青色 '#ba55d3'淡紫色
# 颜色支持 hex 格式（如 '#1f77b4'）或 matplotlib 颜色名
SOL_ALGOS = [
    ('SPP-SF',        '#1f77b4'),  # 蓝色 - SPP 单站基准
    ('SPP-HFUC',        '#2ca02c'),  # 绿色 - SPP+UDUC
    ('SPP-HFUC-RM',     '#ff7f0e'),  # 橙色 - SPP+UDUC+RM
    # ('SPP-HFUC-RM-IGG3', '#00FFFF'),  # 青色 - SPP+UDUC+RM+IGG3
]

# ---- 数据集名称 ----
DATA_SET = 'train'  # 数据集名称 (train, test 等)

# ---- 可视化范围筛选（空列表表示分析全部数据集）----
# 例: SELECTED_DATASETS = ['2022-02-24-18-29-us-ca-lax-o', '2023-05-26-18-50-us-ca-sjc-ge2' 2020-12-10-22-52-us-ca-sjc-c'] 
SELECTED_DATASETS = [] # 空列表表示分析全部

# ====== 固定配置参数 ======
DRAW_SET = 'picture'
REPO_SET = 'report'
DATA_DIR = '../data/'

DRAW_DIR = abspath(DATA_DIR + DRAW_SET)
REPO_DIR = abspath(DATA_DIR + REPO_SET)
today_str = date.today().strftime("%Y%m%d")

# 设置学术级绘图参数
size_font = 10

plt.style.use(['science', 'no-latex'])
plt.rcParams['font.family'] = ['Times New Roman', 'SimSun']
plt.rcParams.update({
    'font.size': size_font,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'grid.color': 'gray',
    'grid.linewidth': 0.5,
    'axes.linewidth': 1.0,
})

fig_width_mm = [75, 90, 120, 150]
fig_height_mm = [60, 80, 100, 120]
mm = 1 / 25.4
fig_width = [w * mm for w in fig_width_mm]
fig_height = [h * mm for h in fig_height_mm]

print('='*70)
print('定位结果分析系统（多算法对比）')
print('='*70)
print(f'数据目录: {DATA_DIR}')
print(f'数据集: {DATA_SET}')
print(f'待比较算法: {[a[0] for a in SOL_ALGOS]}')
print(f'数据集筛选: {"全部" if not SELECTED_DATASETS else SELECTED_DATASETS}')
print(f'分析日期: {date.today().strftime("%Y-%m-%d")}')

In [ ]:
# 加载所有算法的定位结果文件

algo_data = {}
print('算法文件检查:')
for tag, color in SOL_ALGOS:
    pattern = join(DATA_DIR, f'locations_{tag}_{DATA_SET}_*.csv')
    files = sorted(glob(pattern), reverse=True)
    if files:
        algo_data[tag] = {'file': files[0], 'df': None}
        print(f'  {tag}: {os.path.basename(files[0])} \u2713')
    else:
        print(f'  {tag}: 未找到文件 \u2717')

groundtruth_file = join(DATA_DIR, f'ground_truths_{DATA_SET}.csv')
print(f'\n地面真实: {os.path.basename(groundtruth_file)} {"\u2713" if os.path.isfile(groundtruth_file) else "\u2717"}')

if not algo_data:
    print('\n警告: 没有找到任何算法的定位结果文件\uff01')
elif not os.path.isfile(groundtruth_file):
    print('\n警告: 地面真值文件不存在\uff01')
else:
    print('\n\u2713 所有输入文件已就绪')

In [ ]:
# 依次读取各算法数据（保留原始列名，algo 标识在合并阶段添加）
for tag, info in algo_data.items():
    info['df'] = pd.read_csv(info['file'])

print('各算法数据概况:')
for tag, info in algo_data.items():
    df = info['df']
    print(f'  {tag}: {len(df)} 行, 列名: {list(df.columns[:6])}...')

sample_tag = list(algo_data.keys())[0]
print(f'\n样本数据 ({sample_tag}):')
print(algo_data[sample_tag]['df'].head(3))

In [ ]:
# 加载地面真实数据
df_truth = pd.read_csv(groundtruth_file)
print(f'地面真实数据:')
print(f'  行数: {len(df_truth)}')
print(f'  列名: {list(df_truth.columns)}')
print(f'\n样本数据:')
print(df_truth.head(3))

## 第2部分：数据预处理与合并

In [ ]:
# 定义绑一列名并合并所有算法数据
loc_cols = ['tripId', 'UnixTimeMillis', 'Lat_calc', 'Lon_calc', 'Height_calc',
            'Quality', 'NumSatellites', 'Sde', 'Sdu', 'Sdn', 'Sdne', 'Sdeu', 'Sdun']

df_list = []
for tag, info in algo_data.items():
    df = info['df'].copy()
    df.columns = loc_cols[:len(df.columns)]
    df['algo'] = tag
    df_list.append(df)

df_loc_all = pd.concat(df_list, ignore_index=True)
print(f'合并后总行数: {len(df_loc_all)}')

df_truth_renamed = df_truth.copy()
df_truth_renamed.columns = [col.strip() for col in df_truth_renamed.columns]
df_truth_renamed.columns = ['tripId', 'UnixTimeMillis', 'Lat_truth', 'Lon_truth', 'Height_truth']

df_merged = pd.merge(df_loc_all, df_truth_renamed, on=['tripId', 'UnixTimeMillis'], how='inner')

print(f'数据合并完成:')
print(f'  定位数据行数: {len(df_loc_all)}')
print(f'  真实数据行数: {len(df_truth)}')
print(f'  合并后行数: {len(df_merged)}')
print(f'  匹配率: {100*len(df_merged)/max(len(df_loc_all), len(df_truth)):.1f}%')

In [ ]:
# 分类信息提取
df_merged['phone'] = df_merged['tripId'].apply(lambda x: x.split('/')[-1])
df_merged['dataset'] = df_merged['tripId'].apply(lambda x: '/'.join(x.split('/')[:-1]))

print(f'分类信息提取:')
print(f'  Dataset 数量: {df_merged["dataset"].nunique()}')
print(f'  Phone 设备数量: {df_merged["phone"].nunique()}')
print(f'  Algorithm 数量: {df_merged["algo"].nunique()}')

数据集筛选（可选）

In [ ]:
# 根据 SELECTED_DATASETS 筛选数据（空列表表示全部保留）
if SELECTED_DATASETS:
    n_before = len(df_merged)
    df_merged = df_merged[df_merged['dataset'].isin(SELECTED_DATASETS)]
    print(f'数据集筛选: {n_before} -> {len(df_merged)} 行')
    print(f'  保留数据集: {sorted(df_merged["dataset"].unique())}')
    if len(df_merged) == 0:
        print('\n警告: 筛选后无数据\uff01请检查 SELECTED_DATASETS 配置\u3002')
else:
    print(f'数据集筛选: 全部 {df_merged["dataset"].nunique()} 个 Dataset 纳入分析')
    print(f'  数据集列表: {sorted(df_merged["dataset"].unique())}')

## 第3部分：坐标转换与误差计算

In [ ]:
# 坐标转换：WGS84转ENU
a = 6378137.0
e2 = 0.081819189

def lat_lon_to_enu(lat_ref, lon_ref, h_ref, lat, lon, h):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    lat_ref_rad = np.radians(lat_ref)
    lon_ref_rad = np.radians(lon_ref)
    N = a / np.sqrt(1 - e2 * np.sin(lat_ref_rad)**2)
    X_ref = (N + h_ref) * np.cos(lat_ref_rad) * np.cos(lon_ref_rad)
    Y_ref = (N + h_ref) * np.cos(lat_ref_rad) * np.sin(lon_ref_rad)
    Z_ref = (N * (1 - e2) + h_ref) * np.sin(lat_ref_rad)
    N_t = a / np.sqrt(1 - e2 * np.sin(lat_rad)**2)
    X = (N_t + h) * np.cos(lat_rad) * np.cos(lon_rad)
    Y = (N_t + h) * np.cos(lat_rad) * np.sin(lon_rad)
    Z = (N_t * (1 - e2) + h) * np.sin(lat_rad)
    dX, dY, dZ = X - X_ref, Y - Y_ref, Z - Z_ref
    sin_lat, cos_lat = np.sin(lat_ref_rad), np.cos(lat_ref_rad)
    sin_lon, cos_lon = np.sin(lon_ref_rad), np.cos(lon_ref_rad)
    E = -sin_lon * dX + cos_lon * dY
    N_out = -sin_lat * cos_lon * dX - sin_lat * sin_lon * dY + cos_lat * dZ
    U = cos_lat * cos_lon * dX + cos_lat * sin_lon * dY + sin_lat * dZ
    return E, N_out, U

print('ENU坐标转换函数已定义')

In [ ]:
# 计算 ENU 坐标（对所有算法统一处理）
print('计算 ENU 坐标...')
df_merged[['err_e', 'err_n', 'err_u']] = df_merged.apply(
    lambda row: pd.Series(lat_lon_to_enu(row['Lat_truth'], row['Lon_truth'], row['Height_truth'],
                                    row['Lat_calc'], row['Lon_calc'], row['Height_calc'])), axis=1)

df_merged['err_h'] = np.sqrt(df_merged['err_e']**2 + df_merged['err_n']**2)
df_merged['err_3d'] = np.sqrt(df_merged['err_e']**2 + df_merged['err_n']**2 + df_merged['err_u']**2)

print('\u2713 完成\n')

print(f'各算法 ENU 误差范围:')
for tag in df_merged['algo'].unique():
    sub = df_merged[df_merged['algo'] == tag]
    print(f'  [{tag}]')
    print(f'    East  (E): {sub["err_e"].min():>8.3f} ~ {sub["err_e"].max():>8.3f} m')
    print(f'    North (N): {sub["err_n"].min():>8.3f} ~ {sub["err_n"].max():>8.3f} m')
    print(f'    Up    (U): {sub["err_u"].min():>8.3f} ~ {sub["err_u"].max():>8.3f} m')

In [ ]:
# 综合误差统计（按算法分组打印）
print(f'\n综合误差统计:')
print('='*80)

h_error = df_merged['err_h']
e_error = df_merged['err_e']
n_error = df_merged['err_n']
u_error = df_merged['err_u']

for tag in df_merged['algo'].unique():
    sub = df_merged[df_merged['algo'] == tag]
    h = sub['err_h']
    u = sub['err_u']
    print(f'\n[{tag}] - 样本数: {len(sub)}')
    print(f'  水平误差: RMS={np.sqrt(np.mean(h**2)):.3f}m, Mean={h.mean():.3f}m, Median={h.median():.3f}m, P68={np.percentile(h,68):.3f}m, P95={np.percentile(h,95):.3f}m')
    print(f'  \u22641m: {100*np.sum(h<=1)/len(h):.1f}%  \u22641.5m: {100*np.sum(h<=1.5)/len(h):.1f}%  \u22643m: {100*np.sum(h<=3)/len(h):.1f}%')
    print(f'  垂直误差: RMS={np.sqrt(np.mean(u**2)):.3f}m, Mean={np.abs(u).mean():.3f}m, Median={np.median(np.abs(u)):.3f}m, P68={np.percentile(np.abs(u),68):.3f}m, P95={np.percentile(np.abs(u),95):.3f}m')

## 第4部分：时间序列分析（选择性分析）

In [ ]:
# 时间序列图配置（设置为 None 则不绘制）
# ['2022-02-24-18-29-us-ca-lax-o', '2023-05-26-18-50-us-ca-sjc-ge2' '2020-12-10-22-52-us-ca-sjc-c']
# '2021-01-05-21-52-us-ca-mtv-d' '2021-04-02-20-43-us-ca-mtv-f' '2022-02-24-18-29-us-ca-lax-o'
TIME_SERIES_TRIPS = []
TIME_SERIES_PHONES = [] # ['pixel6pro','s22ultra','mi8']

# 设定颜色映射（按照SOL_ALGOS方式,需保证SOL_ALGOS变量已经在全局定义）
algo_color_map = {k: v for k, v in SOL_ALGOS}

if TIME_SERIES_TRIPS is not None or TIME_SERIES_PHONES is not None:
    print(f'绘制时间序列对比图（多算法）...')
    df_ts = df_merged.copy()
    if TIME_SERIES_TRIPS is not None:
        if isinstance(TIME_SERIES_TRIPS, str): TIME_SERIES_TRIPS = [TIME_SERIES_TRIPS]
        df_ts = df_ts[df_ts['dataset'].isin(TIME_SERIES_TRIPS)]
    if TIME_SERIES_PHONES is not None:
        if isinstance(TIME_SERIES_PHONES, str): TIME_SERIES_PHONES = [TIME_SERIES_PHONES]
        df_ts = df_ts[df_ts['phone'].isin(TIME_SERIES_PHONES)]
    if len(df_ts) == 0:
        print('  警告: 没有符合条件的数据')
    else:
        # 确保时间从0开始
        t_min = df_ts['UnixTimeMillis'].min()
        df_ts['time_sec'] = (df_ts['UnixTimeMillis'] - t_min) / 1000

        for trip_id in df_ts['tripId'].unique():
            df_trip = df_ts[df_ts['tripId'] == trip_id]
            trip_name = trip_id.replace('/', '_').replace(':', '-')
            algo_list = [a for a, _ in SOL_ALGOS if a in df_trip['algo'].unique()]

            # ----------- 水平误差对比 -----------
            fig_h, ax_h = plt.subplots(1, 1, figsize=(fig_width[2]-32*mm, fig_height[0]-10*mm))  # 扁长 fig_width[2]+10*mm, fig_height[0]-10*mm
            for algo in algo_list:
                df_sub = df_trip[df_trip['algo'] == algo]
                ax_h.plot(
                    df_sub['time_sec'],
                    df_sub['err_h'],
                    label=algo,
                    color=algo_color_map.get(algo, None),
                    alpha=0.8,
                    linewidth=1.2
                )
            ax_h.set_ylabel('水平误差 (m)', fontsize=size_font)
            ax_h.set_xlabel('相对时间 (秒)', fontsize=size_font)
            ax_h.set_title(f'{trip_id} - 水平误差(m)', fontsize=size_font)
            ax_h.grid(True, alpha=0.3)
            ax_h.set_ylim(-2, 50)  # 修改上下限
            ax_h.tick_params(axis='both', which='major', labelsize=size_font)
            ax_h.legend(
                loc='upper left',
                framealpha=0.85,
                frameon=True,
                # ncol=2,
                edgecolor='k',
                handlelength=1.5,
                handletextpad=0.2,
                borderpad=0.3,
                fontsize=8.5
            )
            plt.tight_layout()
            plt.savefig(join(DRAW_DIR, f'00_time_series_compare_h_{trip_name}_{today_str}.png'), dpi=300, bbox_inches='tight')
            plt.show()

            # ----------- 高程误差对比 -----------
            fig_u, ax_u = plt.subplots(1, 1, figsize=(fig_width[2]-32*mm, fig_height[0]-10*mm))  # fig_width[2]+10*mm, fig_height[0]-10*mm
            for algo in algo_list:
                df_sub = df_trip[df_trip['algo'] == algo]
                ax_u.plot(
                    df_sub['time_sec'],
                    df_sub['err_u'],
                    label=algo,
                    color=algo_color_map.get(algo, None),
                    alpha=0.8,
                    linewidth=1.2
                )
            ax_u.set_ylabel('高程误差 (m)', fontsize=size_font)
            ax_u.set_xlabel('相对时间 (秒)', fontsize=size_font)
            ax_u.set_title(f'{trip_id} - 高程误差(m)', fontsize=size_font)
            ax_u.grid(True, alpha=0.3)
            ax_u.set_ylim(-50, 100)  # 修改上下限
            ax_u.tick_params(axis='both', which='major', labelsize=size_font)
            ax_u.legend(
                loc='upper left',
                framealpha=0.85,
                frameon=True,
                # ncol=2,
                edgecolor='k',
                handlelength=1.5,
                handletextpad=0.2,
                borderpad=0.3,
                fontsize=8.5
            )
       
            plt.tight_layout()
            plt.savefig(join(DRAW_DIR, f'00_time_series_compare_u_{trip_name}_{today_str}.png'), dpi=300, bbox_inches='tight')
            plt.show()

            # ----------- 可见卫星数 -----------
            sat_algo = None
            for algo in algo_list:
                if not df_trip[df_trip['algo'] == algo].empty:
                    sat_algo = algo
                    break
            if sat_algo is not None:
                df_sat = df_trip[df_trip['algo'] == sat_algo]
                fig_sat, ax_sat = plt.subplots(1, 1, figsize=(fig_width[2]-30*mm, fig_height[0]-20*mm)) #fig_width[2]+10*mm, fig_height[0]-20*mm
                ax_sat.plot(
                    df_sat['time_sec'],
                    df_sat['NumSatellites'],
                    label='可见卫星数',
                    color='#1f77b4',  # 可单独指定颜色
                    linewidth=1.3,
                    alpha=0.85
                )
                ax_sat.set_ylabel('可见卫星数', fontsize=size_font)
                ax_sat.set_xlabel('相对时间 (秒)', fontsize=size_font)
                ax_sat.set_title(f'{trip_id} - 可见卫星数', fontsize=size_font)
                ax_sat.grid(True, alpha=0.3)
                ax_sat.tick_params(axis='both', which='major', labelsize=size_font)
                ax_sat.legend(
                    loc='lower left',
                    framealpha=0.85,
                    frameon=True,
                    edgecolor='k',
                    handlelength=1.5,
                    handletextpad=0.2,
                    borderpad=0.3,
                    fontsize=size_font
                )
                plt.tight_layout()
                plt.savefig(join(DRAW_DIR, f'00_time_series_satellite_{trip_name}_{today_str}.png'), dpi=300, bbox_inches='tight')
                plt.show()

        print(f'分算法对比的时间序列图绘制完成\n')
else:
    print('时间序列图已跳过（TIME_SERIES_TRIPS 未配置）\n')

## 第5部分：分组统计分析

In [ ]:
# 只统计 H 和 U 方向精度指标
def calculate_statistics(df_group):
    results = {}
    err_h = df_group['err_h'].values
    err_u = df_group['err_u'].values

    # H方向
    results['RMS_H'] = round(np.sqrt(np.mean(err_h ** 2)), 2)
    results['Mean_H'] = round(np.mean(np.abs(err_h)), 2)        # 只要绝对误差
    results['Median_H'] = round(np.median(np.abs(err_h)), 2)
    results['Std_H'] = round(np.std(err_h), 2)
    results['P68_H'] = round(np.percentile(np.abs(err_h), 68), 2)
    results['P95_H'] = round(np.percentile(np.abs(err_h), 95), 2)
    results['Within_1m_H'] = round(100 * np.sum(np.abs(err_h) <= 1) / len(err_h), 2)
    results['Within_3m_H'] = round(100 * np.sum(np.abs(err_h) <= 3) / len(err_h), 2)
    # RMS.68_H/RMS.95_H 只对在该范围内的点求RMS
    m68h = np.abs(err_h) <= results['P68_H']
    m95h = np.abs(err_h) <= results['P95_H']
    results['RMS.68_H'] = round(np.sqrt(np.mean(err_h[m68h] ** 2)), 2) if m68h.sum() > 0 else np.nan
    results['RMS.95_H'] = round(np.sqrt(np.mean(err_h[m95h] ** 2)), 2) if m95h.sum() > 0 else np.nan

    # U方向
    results['RMS_U'] = round(np.sqrt(np.mean(err_u ** 2)), 2)
    results['Mean_U'] = round(np.mean(np.abs(err_u)), 2)        # 只要绝对误差
    results['Median_U'] = round(np.median(np.abs(err_u)), 2)
    results['Std_U'] = round(np.std(err_u), 2)
    results['P68_U'] = round(np.percentile(np.abs(err_u), 68), 2)
    results['P95_U'] = round(np.percentile(np.abs(err_u), 95), 2)
    results['Within_1m_U'] = round(100 * np.sum(np.abs(err_u) <= 1) / len(err_u), 2)
    results['Within_3m_U'] = round(100 * np.sum(np.abs(err_u) <= 3) / len(err_u), 2)
    m68u = np.abs(err_u) <= results['P68_U']
    m95u = np.abs(err_u) <= results['P95_U']
    results['RMS.68_U'] = round(np.sqrt(np.mean(err_u[m68u] ** 2)), 2) if m68u.sum() > 0 else np.nan
    results['RMS.95_U'] = round(np.sqrt(np.mean(err_u[m95u] ** 2)), 2) if m95u.sum() > 0 else np.nan

    results['Count'] = len(df_group)
    return results

print('误差指标计算函数已定义')

### 5.1 Dataset

In [ ]:
# 按 Dataset 分组统计（Dataset x Algorithm 交叉统计）
print(f'按 Dataset 分组统计（各算法）')
print('='*80)

dataset_stats = []
for dataset in sorted(df_merged['dataset'].unique()):
    dataset_df = df_merged[df_merged['dataset'] == dataset]
    for algo in sorted(dataset_df['algo'].unique()):
        algo_df = dataset_df[dataset_df['algo'] == algo]
        stats = calculate_statistics(algo_df)
        stats['Dataset'] = dataset
        stats['Algorithm'] = algo
        dataset_stats.append(stats)

df_dataset_stats = pd.DataFrame(dataset_stats)
# 只统计 H 和 U（垂直方向）相关指标，并参照计算函数，仅保留这些列
cols = [
    'Dataset', 'Algorithm', 'Count',
    'RMS_H', 'RMS_U',
    'Mean_H', 'Mean_U',
    'Median_H', 'Median_U',
    'Std_H', 'Std_U',
    'P68_H', 'P68_U',
    'P95_H', 'P95_U',
    'RMS.68_H', 'RMS.68_U',
    'RMS.95_H', 'RMS.95_U'
    'Within_1m_H', 'Within_1m_U',
    'Within_3m_H', 'Within_3m_U',

]
df_dataset_stats = df_dataset_stats[[c for c in cols if c in df_dataset_stats.columns]]
print(df_dataset_stats.to_string(index=False))

### 5.2 Phone

In [ ]:
# 按 Phone 分组统计（Phone x Algorithm 交叉统计），仅统计 H 和 U 方向，新的精度指标参照上个代码块
print(f'按 Phone 分组统计（各算法，仅H和U方向）')
print('='*80)

phone_stats = []
for phone in sorted(df_merged['phone'].unique()):
    phone_df = df_merged[df_merged['phone'] == phone]
    for algo in sorted(phone_df['algo'].unique()):
        algo_df = phone_df[phone_df['algo'] == algo]
        stats = calculate_statistics(algo_df)
        stats['Phone'] = phone
        stats['Algorithm'] = algo
        phone_stats.append(stats)

df_phone_stats = pd.DataFrame(phone_stats)
# 仅统计 H 和 U（垂直方向）相关指标，参照上个代码块
cols = [
    'Phone', 'Algorithm', 'Count',
    'RMS_H', 'RMS_U',
    'Mean_H', 'Mean_U',
    'Median_H', 'Median_U',
    'Std_H', 'Std_U',
    'P68_H', 'P68_U',
    'P95_H', 'P95_U',
    'RMS.68_H', 'RMS.68_U',
    'RMS.95_H', 'RMS.95_U',
    'Within_1m_H', 'Within_1m_U',
    'Within_3m_H', 'Within_3m_U',
]
df_phone_stats = df_phone_stats[[c for c in cols if c in df_phone_stats.columns]]
print(df_phone_stats.to_string(index=False))

### 5.3 Dataset x Phone x Algorithm

In [ ]:
# 按数据文件为单位统计（加入 Algorithm 维度）
print(f'按数据文件统计分析...（各算法，仅H和U方向）')
print('='*80)

file_stats = []
for trip_id in df_merged['tripId'].unique():
    df_file = df_merged[df_merged['tripId'] == trip_id]
    phone = trip_id.split('/')[-1]
    dataset = '/'.join(trip_id.split('/')[:-1])
    for algo in sorted(df_file['algo'].unique()):
        algo_df = df_file[df_file['algo'] == algo]
        stats = calculate_statistics(algo_df)
        stats['tripId'] = trip_id
        stats['phone'] = phone
        stats['dataset'] = dataset
        stats['algo'] = algo
        file_stats.append(stats)

df_file_stats = pd.DataFrame(file_stats)
# 仅统计 H 和 U 方向的参数，参照 cell 22 的 cols
cols = [
    'tripId', 'dataset', 'phone', 'algo', 'Count',
    'RMS_H', 'RMS_U',
    'Mean_H', 'Mean_U',
    'Median_H', 'Median_U',
    'Std_H', 'Std_U',
    'P68_H', 'P68_U',
    'P95_H', 'P95_U',
    'RMS.68_H', 'RMS.68_U',
    'RMS.95_H', 'RMS.95_U',
    'Within_1m_H', 'Within_1m_U',
    'Within_3m_H', 'Within_3m_U',
]
df_file_stats = df_file_stats[[c for c in cols if c in df_file_stats.columns]]
print(df_file_stats.to_string(index=False))
file_stats_csv = f'statistics_{DATA_SET}_{today_str}.csv'
df_file_stats.to_csv(join(REPO_DIR, file_stats_csv), index=False)
print(f'\n文件统计已保存: {file_stats_csv}')

## 第6部分：可视化图表

### 6.1 CDF 多算法误差对比

#### 6.1.1 整体分析

In [ ]:
# 绘制多算法 CDF 对比图
fig, axes = plt.subplots(1, 2, figsize=(fig_width[3], fig_height[0]))

def plot_cdf_multi(ax, data_dict, label, x_max=None):
    for tag, color in SOL_ALGOS:
        if tag in data_dict:
            err = data_dict[tag]
            sd = np.sort(err); cdf = np.arange(1, len(sd)+1)/len(sd)
            ax.plot(sd, cdf, color=color, linewidth=2, label=tag)
    ax.set_xlabel('绝对误差 (m)')
    ax.set_ylabel('累计概率')
    ax.set_title(f'{label}', fontsize=size_font)
    ax.grid(True, alpha=0.3)
    ax.legend(framealpha=0.8, frameon=True, edgecolor='k', handlelength=1.5, handletextpad=0.1, borderpad=0.3)
    ax.set_ylim(0, 1.05)
    if x_max:
        ax.set_xlim(0, x_max)
    # 绘制PCT68和PCT95的横线，颜色分别用绿色和蓝色
    pct_lines = [
        (0.68, 'green', '0.68', 'top'),
        (0.95, 'blue', '0.95', 'bottom'),
    ]
    for y, color, pct_label, v_align in pct_lines:
        ax.axhline(y, color=color, linestyle='--', linewidth=1)
        # 标签位置
        if v_align == 'top':
            ax.text(ax.get_xlim()[1]*0.15, y, pct_label, va='bottom', ha='right', color=color)
        else:  # bottom
            ax.text(ax.get_xlim()[1]*0.15, y-0.02, pct_label, va='top', ha='right', color=color)

h_data = {tag: df_merged[df_merged['algo']==tag]['err_h'].values for tag,_ in SOL_ALGOS if tag in df_merged['algo'].values}
u_data = {tag: np.abs(df_merged[df_merged['algo']==tag]['err_u'].values) for tag,_ in SOL_ALGOS if tag in df_merged['algo'].values}

plot_cdf_multi(axes[0], h_data, '水平方向误差', x_max=20)
plot_cdf_multi(axes[1], u_data, '垂直方向误差', x_max=30)

plt.tight_layout()
algo_str = '_'.join([a[0] for a in SOL_ALGOS])
plt.savefig(join(DRAW_DIR, f'01_cdf_{algo_str}_{DATA_SET}_{today_str}.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'已保存: 01_cdf_{DATA_SET}_{algo_str}_{today_str}.png')

#### 6.1.2 Phone

In [ ]:
# 为每个 phone 分别绘制 CDF 图，横向对比不同算法（不再标注P68/P95等统计数据）
for phone in sorted(df_merged['phone'].unique()):
    fig, axes = plt.subplots(1, 2, figsize=(fig_width[1]-8*mm, fig_height[0]-25*mm)) #  fig_width[3]-5*mm, fig_height[0]-5*mm

    def plot_cdf_multi_algo_by_phone(ax, data_dict, label, x_max=None, show_ylabel=True):
        for tag, color in SOL_ALGOS:
            if tag in data_dict:
                err = data_dict[tag]
                if len(err) == 0:
                    continue
                sd = np.sort(err)
                cdf = np.arange(1, len(sd)+1)/len(sd)
                ax.plot(sd, cdf, color=color, linewidth=2, label=tag)
        # 字体适中调大
        label_fontsize = 9 #14
        title_fontsize = 9 #16
        legend_fontsize = 7 #12
        tick_fontsize = 9 #12
        ax.set_xlabel('绝对误差 (m)', fontsize=label_fontsize)
        if show_ylabel:
            ax.set_ylabel('累计概率', fontsize=label_fontsize)
        else:
            ax.set_ylabel('')
        ax.set_title(f'{label}', fontsize=title_fontsize)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
        if x_max:
            ax.set_xlim(0, x_max)
        ax.axhline(0.95, color='black', linestyle='--', linewidth=1)
        # 让"0.95"稍微远离虚线，提高可读性
        ax.text(ax.get_xlim()[1]*0.2, 0.92, '0.95', va='top', ha='right', color='black', fontsize=label_fontsize)
 
        ax.tick_params(axis='both', which='major', labelsize=tick_fontsize)
        ax.legend(framealpha=0.8, frameon=True, edgecolor='k',
                  handlelength=1.5, handletextpad=0.1, borderpad=0.3,
                  loc='lower right', fontsize=legend_fontsize)

    # 获取当前phone下不同算法的误差数据
    h_data = {tag: df_merged[(df_merged['phone']==phone) & (df_merged['algo']==tag)]['err_h'].values for tag,_ in SOL_ALGOS}
    u_data = {tag: np.abs(df_merged[(df_merged['phone']==phone) & (df_merged['algo']==tag)]['err_u'].values) for tag,_ in SOL_ALGOS}

    plot_cdf_multi_algo_by_phone(axes[0], h_data, f'{phone} 水平方向误差', x_max=15, show_ylabel=True)
    plot_cdf_multi_algo_by_phone(axes[1], u_data, f'{phone} 垂直方向误差', x_max=30, show_ylabel=False)

    # 采用gridspec，进一步缩小左右图间距
    fig.subplots_adjust(wspace=0.03)  # 极小的wspace
    plt.tight_layout(rect=[0, 0, 1, 1], pad=0.3) # pad调小
    plt.savefig(join(DRAW_DIR, f'01_cdf_phone_{DATA_SET}_{phone}_{today_str}.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'已保存: 01_cdf_phone_{DATA_SET}_{phone}_{today_str}.png')

### 6.2 直方图 多算法误差分布对比

#### 6.2.1 整体分析

In [ ]:
# 多算法误差分布直方图（水平 + 垂直，2行1列）
fig, axes = plt.subplots(2, 1, figsize=(fig_width[2], fig_height[2]-10*mm))

# 统计各算法的中位数（用于图例）
median_h_dict = {}
median_u_dict = {}

# 动态透明度: 底层最不透明，顶层最透明（越后出现的透明度越高）
n_algos_h = sum([tag in df_merged['algo'].values for tag, _ in SOL_ALGOS])
n_algos_u = n_algos_h  # 假设数量一致

# 生成透明度递减的列表（底层最低，顶层最高）
start_alpha, end_alpha = 0.4, 0.1
if n_algos_h > 1:
    alpha_list_h = np.linspace(start_alpha, end_alpha, n_algos_h)
else:
    alpha_list_h = [start_alpha]
if n_algos_u > 1:
    alpha_list_u = np.linspace(start_alpha, end_alpha, n_algos_u)
else:
    alpha_list_u = [start_alpha]

# 水平误差
algo_valid_h = [ (tag, color) for tag, color in SOL_ALGOS if tag in df_merged['algo'].values ]
for idx, (item, hist_alpha) in enumerate(zip(algo_valid_h, alpha_list_h)):
    tag, color = item
    err_h = df_merged[df_merged['algo']==tag]['err_h']
    xmax = 10
    bins = np.linspace(0, xmax, 300)
    axes[0].hist(err_h, bins=bins, color=color, alpha=hist_alpha, label=tag, density=True)
    kde = gaussian_kde(err_h); x = np.linspace(0, xmax, 500)
    kde_y = kde(x)
    axes[0].plot(x, kde_y, color=color, lw=2, alpha=0.8)
    # 保存中位数
    median_h = np.median(err_h)
    median_h_dict[tag] = median_h
    # 中位数线：仅从曲线顶部到底部（x轴）
    kde_median_val = kde(median_h)
    axes[0].vlines(median_h, 0, kde_median_val, color=color, lw=1.6, ls='--', alpha=0.85, zorder=10)

axes[0].set_title('水平误差分布对比', fontsize=size_font)
axes[0].set_ylabel('概率密度')
# 构造带中位数信息的图例
legend_labels_h = [f'{tag} ({median_h_dict[tag]:.2f}m)' for tag, _ in SOL_ALGOS if tag in median_h_dict]
legend_handles_h = []
for idx, (tag, color) in enumerate(SOL_ALGOS):
    if tag in median_h_dict:
        handle = plt.Rectangle((0,0),1,1, color=color, alpha=1.0)  # 图例不要透明
        legend_handles_h.append(handle)
axes[0].legend(legend_handles_h, legend_labels_h, framealpha=0)
axes[0].set_xlim(0, axes[0].get_xlim()[1])
axes[0].yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
axes[0].ticklabel_format(axis='y', style='sci', scilimits=(0,0))

# 垂直误差
algo_valid_u = [ (tag, color) for tag, color in SOL_ALGOS if tag in df_merged['algo'].values ]
for idx, (item, hist_alpha) in enumerate(zip(algo_valid_u, alpha_list_u)):
    tag, color = item
    err_u = np.abs(df_merged[df_merged['algo']==tag]['err_u'])
    xmax = 20
    bins = np.linspace(0, xmax, 300)
    axes[1].hist(err_u, bins=bins, color=color, alpha=hist_alpha, label=tag, density=True)
    kde = gaussian_kde(err_u); x = np.linspace(0, xmax, 300)
    kde_y = kde(x)
    axes[1].plot(x, kde_y, color=color, lw=2, alpha=0.8)
    # 保存中位数
    median_u = np.median(err_u)
    median_u_dict[tag] = median_u
    # 中位数线：仅从曲线顶部到底部（x轴）
    kde_median_val = kde(median_u)
    axes[1].vlines(median_u, 0, kde_median_val, color=color, lw=1.6, ls='--', alpha=0.85, zorder=10)

axes[1].set_title('垂直误差分布对比', fontsize=size_font)
axes[1].set_xlabel('绝对误差 (m)')
axes[1].set_ylabel('概率密度')
# 构造带中位数信息的图例
legend_labels_u = [f'{tag} ({median_u_dict[tag]:.2f}m)' for tag, _ in SOL_ALGOS if tag in median_u_dict]
legend_handles_u = []
for idx, (tag, color) in enumerate(SOL_ALGOS):
    if tag in median_u_dict:
        handle = plt.Rectangle((0,0),1,1, color=color, alpha=1.0)  # 图例不要透明
        legend_handles_u.append(handle)
axes[1].legend(legend_handles_u, legend_labels_u, framealpha=0)
axes[1].set_xlim(0, axes[1].get_xlim()[1])
axes[1].yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0,0))

plt.tight_layout()
algo_str = '_'.join([a[0] for a in SOL_ALGOS])
plt.savefig(join(DRAW_DIR, f'03_histogram_{DATA_SET}_{algo_str}_{today_str}.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'已保存: 03_histogram_{DATA_SET}_{algo_str}_{today_str}.png')

#### 6.2.2 Phone

In [ ]:
# 按照模版画出每个手机的误差分布直方图，左右排版，字体调大，确保x轴长度远大于y轴长度，y轴高度再短一半

phones_order = ['mi8','pixel6pro','pixel7','pixel7pro','s20ultra','s21ultra','s22ultra']
phones = [p for p in phones_order if p in df_merged['phone'].values]
n_phones = len(phones)
# figsize: 宽1000, 高420 => y轴高度短。进一步让高减半
fig_width_each = 3.3 #6  # 每个手机的单图宽度
fig_height_each = 1.5  # 单子图原来是4，现在减半为2   2.5

# 水平误差 & 垂直误差，左右排版
for phone in phones:
    fig, axes = plt.subplots(1, 2, figsize=(fig_width_each * 2, fig_height_each))  # 1行2列
    median_h_dict = {}
    median_u_dict = {}

    # 动态透明度
    n_algos = sum([tag in df_merged[(df_merged['phone']==phone)]['algo'].values for tag, _ in SOL_ALGOS])
    start_alpha, end_alpha = 0.4, 0.1
    if n_algos > 1:
        alpha_list = np.linspace(start_alpha, end_alpha, n_algos)
    else:
        alpha_list = [start_alpha]

    # 水平误差
    algo_valid_h = [(tag, color) for tag, color in SOL_ALGOS if tag in df_merged[(df_merged['phone']==phone)]['algo'].values]
    for idx, (item, hist_alpha) in enumerate(zip(algo_valid_h, alpha_list)):
        tag, color = item
        err_h = df_merged[(df_merged['algo']==tag) & (df_merged['phone']==phone)]['err_h']
        xmax = 10
        bins = np.linspace(0, xmax, 200)
        axes[0].hist(err_h, bins=bins, color=color, alpha=hist_alpha, label=tag, density=True)
        kde = gaussian_kde(err_h); x = np.linspace(0, xmax, 400)
        kde_y = kde(x)
        axes[0].plot(x, kde_y, color=color, lw=2, alpha=0.8)
        median_h = np.median(err_h)
        median_h_dict[tag] = median_h
        # 只画从曲线顶部到底部（x轴）的中位数竖线
        # (1) 计算在中位数处的最大概率密度
        kde_median_val = kde(median_h)
        axes[0].vlines(median_h, 0, kde_median_val, color=color, lw=1.6, ls='--', alpha=0.85, zorder=10)
    axes[0].set_title(f'{phone} - 水平误差分布', fontsize=10)
    axes[0].set_xlabel('绝对误差 (m)', fontsize=10)
    axes[0].set_ylabel('概率密度', fontsize=10)
    legend_labels_h = [f'{tag} ({median_h_dict[tag]:.2f}m)' for tag in median_h_dict]
    legend_handles_h = [plt.Rectangle((0,0),1,1,color=color,alpha=1.0) 
                        for tag, color in SOL_ALGOS if tag in median_h_dict]
    axes[0].legend(legend_handles_h, legend_labels_h, framealpha=0, fontsize=8)
    axes[0].set_xlim(0, xmax)
    axes[0].set_ylim(bottom=0)
    axes[0].yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
    axes[0].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[0].tick_params(axis='both', labelsize=10)

    # 垂直误差
    algo_valid_u = [(tag, color) for tag, color in SOL_ALGOS if tag in df_merged[(df_merged['phone']==phone)]['algo'].values]
    for idx, (item, hist_alpha) in enumerate(zip(algo_valid_u, alpha_list)):
        tag, color = item
        err_u = np.abs(df_merged[(df_merged['algo']==tag) & (df_merged['phone']==phone)]['err_u'])
        xmax = 20
        bins = np.linspace(0, xmax, 200)
        axes[1].hist(err_u, bins=bins, color=color, alpha=hist_alpha, label=tag, density=True)
        kde = gaussian_kde(err_u); x = np.linspace(0, xmax, 300)
        kde_y = kde(x)
        axes[1].plot(x, kde_y, color=color, lw=2, alpha=0.8)
        median_u = np.median(err_u)
        median_u_dict[tag] = median_u
        # 只画从曲线顶部到底部（x轴）的中位数竖线
        kde_median_val = kde(median_u)
        axes[1].vlines(median_u, 0, kde_median_val, color=color, lw=1.6, ls='--', alpha=0.85, zorder=10)
    axes[1].set_title(f'{phone} - 垂直误差分布', fontsize=10)
    axes[1].set_xlabel('绝对误差 (m)', fontsize=10)
    # axes[1].set_ylabel('概率密度', fontsize=14)
    legend_labels_u = [f'{tag} ({median_u_dict[tag]:.2f}m)' for tag in median_u_dict]
    legend_handles_u = [plt.Rectangle((0,0),1,1,color=color,alpha=1.0) 
                        for tag, color in SOL_ALGOS if tag in median_u_dict]
    axes[1].legend(legend_handles_u, legend_labels_u, framealpha=0, fontsize=8)
    axes[1].set_xlim(0, xmax)
    axes[1].set_ylim(bottom=0)
    axes[1].yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
    axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    axes[1].tick_params(axis='both', labelsize=10)

    plt.tight_layout()
    algo_str = '_'.join([a[0] for a in SOL_ALGOS])
    fname = f"03_hist_{DATA_SET}_{phone}_{algo_str}_{today_str}.png"
    plt.savefig(join(DRAW_DIR, fname), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'已保存: {fname}')

### 6.3 箱线图 多算法误差对比

In [ ]:
# 各手机下，不同算法误差箱线图对比，横轴为手机，不同算法用不同颜色表示
# 指定手机显示顺序
ordered_phones = ['mi8','pixel6pro','pixel7','pixel7pro','s20ultra','s21ultra','s22ultra']
phones = [p for p in ordered_phones if p in df_merged['phone'].values]

algos = [a[0] for a in SOL_ALGOS if a[0] in df_merged['algo'].values]
algo_colors = {a[0]: a[1] for a in SOL_ALGOS if a[0] in algos}

fig, axes = plt.subplots(2, 1, figsize=(fig_width[3], fig_height[1]+5*mm), sharex=True)

# --- 1. 水平误差 Boxplot ---
box_data_h = []
for algo in algos:
    data = [df_merged[(df_merged['phone'] == phone) & (df_merged['algo'] == algo)]['err_h'].values for phone in phones]
    box_data_h.append(data)

box_width = 0.7 / len(algos)  # 箱线图宽度分割

x = np.arange(len(phones))
for i, (algo, data) in enumerate(zip(algos, box_data_h)):
    positions = x - 0.35 + box_width/2 + i*box_width
    bp = axes[0].boxplot(
        data,
        positions=positions,
        widths=box_width*0.9,
        patch_artist=True,
        boxprops=dict(facecolor=algo_colors[algo], color=algo_colors[algo]),
        medianprops=dict(color='black'),
        showfliers=False
    )
axes[0].set_ylabel('水平误差 (m)', fontsize=size_font)
axes[0].set_title('水平误差箱线图', fontsize=size_font)
axes[0].set_xticks(x)
axes[0].set_xticklabels(phones, rotation=30, ha='right')
# axes[0].axhline(y=1, color='orange', linestyle='--', alpha=0.7, label='1m')
# axes[0].axhline(y=3, color='red', linestyle='--', alpha=0.7, label='3m')
# 算法图例
handles = [plt.Rectangle((0,0),1,1, facecolor=algo_colors[algo]) for algo in algos]
axes[0].legend(handles, algos, framealpha=0.8, ncol=3, edgecolor='k')
axes[0].grid(True, alpha=0.3, axis='y')

# --- 2. 垂直误差 Boxplot ---
box_data_u = []
for algo in algos:
    data = [np.abs(df_merged[(df_merged['phone'] == phone) & (df_merged['algo'] == algo)]['err_u'].values) for phone in phones]
    box_data_u.append(data)

for i, (algo, data) in enumerate(zip(algos, box_data_u)):
    positions = x - 0.35 + box_width/2 + i*box_width
    bp = axes[1].boxplot(
        data,
        positions=positions,
        widths=box_width*0.9,
        patch_artist=True,
        boxprops=dict(facecolor=algo_colors[algo], color=algo_colors[algo]),
        medianprops=dict(color='black'),
        showfliers=False
    )
axes[1].set_ylabel('垂直误差 (m)', fontsize=size_font)
axes[1].set_title('垂直误差箱线图', fontsize=size_font)
axes[1].set_xticks(x)
axes[1].set_xticklabels(phones, rotation=30, ha='right')
# axes[1].axhline(y=1, color='orange', linestyle='--', alpha=0.7, label='1m')
# axes[1].axhline(y=3, color='red', linestyle='--', alpha=0.7, label='3m')
axes[1].legend(handles, algos, framealpha=0.8, ncol=3, edgecolor='k', loc='upper left', bbox_to_anchor=(0, 1))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(join(DRAW_DIR, f'04_boxplot_phone_{DATA_SET}_{today_str}.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'已保存: 04_boxplot_phone_{DATA_SET}_{today_str}.png')

### 6.4 柱状图 多算法 RMS 对比

#### 6.4.1 Phone

In [ ]:
# 横轴为手机（固定顺序），不同算法用不同颜色表示的 RMS 柱状图
fig, axes = plt.subplots(2, 1, figsize=(fig_width[3], fig_height[2]-15*mm), sharex=True)

# 指定手机顺序
ordered_phones = ['mi8','pixel6pro','pixel7','pixel7pro','s20ultra','s21ultra','s22ultra']
phone_labels = [p for p in ordered_phones if p in df_merged['phone'].values]
n_phones = len(phone_labels)
algo_labels = [a[0] for a in SOL_ALGOS if a[0] in df_merged['algo'].values]
algo_colors = {a[0]: a[1] for a in SOL_ALGOS if a[0] in df_merged['algo'].values}
n_algos = len(algo_labels)
width = 0.7 / n_algos  # 保证总宽度适中

# 水平 RMS
rms_h_vals = []
for algo in algo_labels:
    rms_for_algo = []
    for phone in phone_labels:
        data = df_merged[(df_merged['algo']==algo) & (df_merged['phone']==phone)]
        val = np.sqrt(np.mean(data['err_h']**2)) if not data.empty else np.nan
        rms_for_algo.append(val)
    rms_h_vals.append(rms_for_algo)

x = np.arange(n_phones)
for i, algo in enumerate(algo_labels):
    offs = x - 0.35 + width/2 + i*width
    bars = axes[0].bar(
        offs, rms_h_vals[i], width, color=algo_colors[algo], label=algo
    )
    for j, (bar, val) in enumerate(zip(bars, rms_h_vals[i])):
        if not np.isnan(val):
            axes[0].text(
                bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, rotation=45
            )

axes[0].set_ylabel('RMSE(m)', fontsize=10)
axes[0].set_title('水平方向误差', fontsize=10)
axes[0].set_xticks(x)
axes[0].set_xticklabels(phone_labels, rotation=30, ha='right', fontsize=10)
axes[0].tick_params(axis='y', labelsize=10)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].legend(framealpha=0.8, edgecolor='k', ncol=2, fontsize=10)
ymax = np.nanmax(rms_h_vals)*1.2 if np.nanmax(rms_h_vals) > 0 else 1
axes[0].set_ylim(0, ymax+4)

# 垂直 RMS
rms_u_vals = []
for algo in algo_labels:
    rms_for_algo = []
    for phone in phone_labels:
        data = df_merged[(df_merged['algo']==algo) & (df_merged['phone']==phone)]
        val = np.sqrt(np.mean(data['err_u']**2)) if not data.empty else np.nan
        rms_for_algo.append(val)
    rms_u_vals.append(rms_for_algo)

for i, algo in enumerate(algo_labels):
    offs = x - 0.35 + width/2 + i*width
    bars = axes[1].bar(
        offs, rms_u_vals[i], width, color=algo_colors[algo], label=algo
    )
    for j, (bar, val) in enumerate(zip(bars, rms_u_vals[i])):
        if not np.isnan(val):
            axes[1].text(
                bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, rotation=30
            )

axes[1].set_ylabel('RMSE(m)', fontsize=10)
axes[1].set_title('垂直方向误差', fontsize=10)
axes[1].set_xticks(x)
axes[1].set_xticklabels(phone_labels, rotation=30, ha='right', fontsize=10)
axes[1].tick_params(axis='y', labelsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend(framealpha=0.8, edgecolor='k', ncol=2, fontsize=10)
ymax = np.nanmax(rms_u_vals)*1.2 if np.nanmax(rms_u_vals) > 0 else 1
axes[1].set_ylim(0, ymax+5)

plt.tight_layout()
plt.savefig(join(DRAW_DIR, f'05_bar_phone_{DATA_SET}_{today_str}.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'已保存: 05_bar_phone_{DATA_SET}_{today_str}.png')

#### 6.4.2 Dataset

In [ ]:
# 手动设定选取哪些dataset（支持"1"、"1-10"、"2,5,7"等形式）
# 例：select_str = "1-10" 只画前10个；select_str = "1,3,7,15-19"画第1,3,7,15,16,17,18,19个
select_str = "0"  # <== 在这里手动调整区间或列表

ordered_datasets = sorted(df_merged['dataset'].unique())

def parse_select_str(select_str, max_len):
    """
    将 '1,2-4,10' 形式的字符串转为索引（从0计数）列表
    """
    sel = []
    items = select_str.split(",")
    for it in items:
        it = it.strip()
        if '-' in it:
            s, e = it.split('-')
            try:
                start = int(s)-1
                end = int(e)
                for i in range(start, end):
                    if 0 <= i < max_len:
                        sel.append(i)
            except:
                continue
        else:
            try:
                idx = int(it)-1
                if 0 <= idx < max_len:
                    sel.append(idx)
            except:
                continue
    # 去重并排序
    return sorted(set(sel))

sel_indices = parse_select_str(select_str, len(ordered_datasets))
if not sel_indices:
    raise ValueError(f"解析select_str失败，当前数据集数量={len(ordered_datasets)}，请修改select_str: '{select_str}'")
dataset_labels = [ordered_datasets[i] for i in sel_indices]

xtick_labels = [str(ds)[:16] for ds in dataset_labels]

n_datasets = len(dataset_labels)
algo_labels = [a[0] for a in SOL_ALGOS if a[0] in df_merged['algo'].values]
algo_colors = {a[0]: a[1] for a in SOL_ALGOS if a[0] in df_merged['algo'].values}
n_algos = len(algo_labels)

# ----------- 关键区段：让x轴各dataset之间绘图留白适中（有留白但更紧凑） -----------
# 设置较小的 spacing_factor 以减小dataset间距，同时保留留白（通常0.6-1.0，有留白但紧凑，原0.7）
spacing_factor = 0.6
group_width = 0.5           # 柱状图一组柱的总宽度（总共spacing_factor的比例被填满）
bar_width = group_width / n_algos   # 每个bar的宽度

x = np.arange(n_datasets) * spacing_factor

label_offset_base = 0.01  # 基础的 y 偏移比例，防止数字重叠

fig, axes = plt.subplots(2, 1, figsize=(max(10, n_datasets*spacing_factor*1.4), fig_height[2]), sharex=True)

# 水平 RMS
rms_h_vals = []
for algo in algo_labels:
    rms_for_algo = []
    for dataset in dataset_labels:
        data = df_merged[(df_merged['algo']==algo) & (df_merged['dataset']==dataset)]
        val = np.sqrt(np.mean(data['err_h']**2)) if not data.empty else np.nan
        rms_for_algo.append(val)
    rms_h_vals.append(rms_for_algo)

for i, algo in enumerate(algo_labels):
    # 使group内各bar居中，组宽为 group_width
    offs = x - group_width/2 + bar_width/2 + i*bar_width
    bars = axes[0].bar(
        offs, rms_h_vals[i], bar_width, color=algo_colors[algo], label=algo
    )
    for j, (bar, val) in enumerate(zip(bars, rms_h_vals[i])):
        # 标签仅在样本较少时显示
        if not np.isnan(val) and n_datasets <= 20:
            ymax = np.nanmax(rms_h_vals)*1.2
            label_offset = label_offset_base * ymax + (n_algos-1-i)*0.03*ymax
            label_y = min(bar.get_height() + label_offset, ymax - 0.03*ymax)
            axes[0].text(
                bar.get_x() + bar.get_width()/2, label_y,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8
            )

axes[0].set_ylabel('RMSE (m)')
axes[0].set_title('水平方向误差 RMSE')
axes[0].set_xticks(x)
axes[0].set_xticklabels(
    xtick_labels,
    rotation=45 if n_datasets<=20 else 90,
    ha='right',
    fontsize=9 if n_datasets<=20 else 7
)
axes[0].grid(True, alpha=0.3, axis='y')
# 横向图例(legend)放在图内顶部中央，稍微下移和图形有间隔
axes[0].legend(
    framealpha=0.8, edgecolor='k',
    loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=n_algos, borderaxespad=0.1
)
ymax = np.nanmax(rms_h_vals)*1.2 if np.nanmax(rms_h_vals) > 0 else 1
axes[0].set_ylim(0, ymax)

# 垂直 RMS
rms_u_vals = []
for algo in algo_labels:
    rms_for_algo = []
    for dataset in dataset_labels:
        data = df_merged[(df_merged['algo']==algo) & (df_merged['dataset']==dataset)]
        val = np.sqrt(np.mean(data['err_u']**2)) if not data.empty else np.nan
        rms_for_algo.append(val)
    rms_u_vals.append(rms_for_algo)

for i, algo in enumerate(algo_labels):
    offs = x - group_width/2 + bar_width/2 + i*bar_width
    bars = axes[1].bar(
        offs, rms_u_vals[i], bar_width, color=algo_colors[algo], label=algo
    )
    for j, (bar, val) in enumerate(zip(bars, rms_u_vals[i])):
        if not np.isnan(val) and n_datasets <= 20:
            ymax = np.nanmax(rms_u_vals)*1.2
            label_offset = label_offset_base * ymax + (n_algos-1-i)*0.03*ymax
            label_y = min(bar.get_height() + label_offset, ymax - 0.03*ymax)
            axes[1].text(
                bar.get_x() + bar.get_width()/2, label_y,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8
            )

axes[1].set_ylabel('RMSE (m)')
axes[1].set_title('垂直方向误差 RMSE')
axes[1].set_xticks(x)
axes[1].set_xticklabels(
    xtick_labels,
    rotation=45 if n_datasets<=20 else 90,
    ha='right',
    fontsize=9 if n_datasets<=20 else 7
)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend(
    framealpha=0.8, edgecolor='k',
    loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=n_algos, borderaxespad=0.1
)
ymax = np.nanmax(rms_u_vals)*1.2 if np.nanmax(rms_u_vals) > 0 else 1
axes[1].set_ylim(0, ymax)

# 图像输出名字增加首末索引信息（1-based）
if sel_indices:
    idx_range_str = f"{sel_indices[0]+1}-{sel_indices[-1]+1}" if len(sel_indices) > 1 else f"{sel_indices[0]+1}"
else:
    idx_range_str = "none"
save_filename = f'05_bar_dataset_{DATA_SET}_{idx_range_str}_{today_str}.png'
plt.tight_layout()
plt.savefig(join(DRAW_DIR, save_filename), dpi=300, bbox_inches='tight')
plt.show()
print(f'已保存: {save_filename}\n当前选取的dataset索引: {sel_indices}\n当前横轴共{n_datasets}个dataset。修改 select_str 可自定义。')

### 6.5 热力图 Dataset x Phone (按算法分页)

In [ ]:
# # Dataset x Phone RMS 热力图（每个算法单独绘制一页）
# pivot_rms_h = {}; pivot_rms_u = {}
# for tag, color in SOL_ALGOS:
#     if tag not in df_merged['algo'].values: continue
#     sub = df_merged[df_merged['algo'] == tag]
#     pivot_rms_h[tag] = sub.groupby(['dataset','phone']).apply(lambda x: np.sqrt(np.mean(x['err_h']**2))).unstack()
#     pivot_rms_u[tag] = sub.groupby(['dataset','phone']).apply(lambda x: np.sqrt(np.mean(x['err_u']**2))).unstack()

# if pivot_rms_h:
#     first_tag = list(pivot_rms_h.keys())[0]  # 改里面序号即可
#     pivot_h = pivot_rms_h[first_tag]; pivot_u = pivot_rms_u[first_tag]
#     fig, axes = plt.subplots(2, 1, figsize=(fig_width[3], 300*mm), constrained_layout=True)
#     for idx, (pivot, title) in enumerate(zip([pivot_h, pivot_u],
#         [f'水平方向误差 RMS ({first_tag})', f'垂直方向误差 RMS ({first_tag})'])):
#         im = axes[idx].imshow(pivot.values, cmap='RdYlGn_r', aspect='auto')
#         axes[idx].set_xticks(np.arange(len(pivot.columns))); axes[idx].set_yticks(np.arange(len(pivot.index)))
#         axes[idx].set_xticklabels(pivot.columns, rotation=45, ha='right')
#         axes[idx].set_yticklabels([str(i)[:16] for i in pivot.index])
#         for i in range(len(pivot.index)):
#             for j in range(len(pivot.columns)):
#                 val = pivot.iloc[i,j]
#                 if not np.isnan(val): axes[idx].text(j, i, f'{val:.1f}', ha='center', va='center')
#         cbar = plt.colorbar(im, ax=axes[idx], fraction=0.045, pad=0.03)
#         cbar.set_label('RMS (m)'); axes[idx].set_title(title)
#         axes[idx].set_xticks(np.arange(-.5, len(pivot.columns), 1), minor=True)
#         axes[idx].set_yticks(np.arange(-.5, len(pivot.index), 1), minor=True)
#         axes[idx].grid(which='minor', color='k', linestyle='-', linewidth=0.5, alpha=0.2)
#         axes[idx].tick_params(which='minor', bottom=False, left=False)
#     plt.savefig(join(DRAW_DIR, f'02_heatmap_{first_tag}_{DATA_SET}_{today_str}.png'), dpi=300, bbox_inches='tight')
#     plt.show()
#     print(f'已保存: 02_heatmap_{first_tag}_{DATA_SET}_{today_str}.png')
#     print(f'注: 共 {len(pivot_rms_h)} 个算法，其他算法热力图可用同样方式扩展')

## 第7部分：生成分析报告

In [ ]:
# 生成综合分析报告（支持多算法）
algo_str = '_'.join([a[0] for a in SOL_ALGOS])
report_file = join(REPO_DIR, f'{algo_str}_report_{DATA_SET}_{today_str}.txt')

def truncated_rms(data, upper_percentile=68):
    """计算截断均方根误差（只考虑从最小到upper_percentile阈值的数据）"""
    threshold = np.percentile(data, upper_percentile)
    truncated = data[data <= threshold]
    if len(truncated) == 0: return np.nan
    return np.sqrt(np.mean(truncated**2))

with open(report_file, 'w', encoding='utf-8') as f:
    f.write('\n'+'='*80+'\n')
    f.write(' ' * 15 + '多算法定位结果分析报告\n')
    f.write('='*80+'\n\n')
    f.write(f'分析日期: {date.today().strftime("%Y-%m-%d")}\n')
    f.write(f'数据集类型: {DATA_SET}\n')
    f.write(f'待比较算法: {[a[0] for a in SOL_ALGOS]}\n')
    f.write(f'样本总数: {len(df_merged):,}\n')
    f.write(f'Dataset 数量: {df_merged["dataset"].nunique()}\n')
    f.write(f'Phone 设备数量: {df_merged["phone"].nunique()}\n')
    f.write(f'Algorithm 数量: {df_merged["algo"].nunique()}\n\n')

    for tag in df_merged['algo'].unique():
        sub = df_merged[df_merged['algo'] == tag]
        h = sub['err_h']
        u = np.abs(sub['err_u'])  # 取绝对值，保证高程误差为正数

        # 水平特征值（E-N 平面）
        rmse_h = np.sqrt(np.mean(h ** 2))
        mean_h = h.mean()
        median_h = h.median()
        std_h = h.std()
        p68_h = np.percentile(h, 68)
        p95_h = np.percentile(h, 95)
        rms68_h = truncated_rms(h, 68)
        rms95_h = truncated_rms(h, 95)
        within_1m_h = (h <= 1).sum() / len(h) * 100
        within_3m_h = (h <= 3).sum() / len(h) * 100

        # 高程方向 (U) - 绝对值
        rmse_u = np.sqrt(np.mean(u ** 2))
        mean_u = u.mean()
        median_u = u.median()
        std_u = u.std()
        p68_u = np.percentile(u, 68)
        p95_u = np.percentile(u, 95)
        rms68_u = truncated_rms(u, 68)
        rms95_u = truncated_rms(u, 95)
        within_1m_u = (u <= 1).sum() / len(u) * 100
        within_3m_u = (u <= 3).sum() / len(u) * 100

        f.write('=' * 80 + f'\n算法: {tag}（样本数: {len(sub):,}）\n' + '=' * 80 + '\n\n')
        f.write('水平误差 (E-N 平面):\n')
        f.write(f'  1. RMS:       {rmse_h:>10.4f} m\n')
        f.write(f'  2. Mean:      {mean_h:>10.4f} m\n')
        f.write(f'  3. Median:    {median_h:>10.4f} m\n')
        f.write(f'  4. Std:       {std_h:>10.4f} m\n')
        f.write(f'  5. P68:       {p68_h:>10.4f} m\n')
        f.write(f'  6. P95:       {p95_h:>10.4f} m\n')
        f.write(f'  7. RMS.68:    {rms68_h:>10.4f} m\n')
        f.write(f'  8. RMS.95:    {rms95_h:>10.4f} m\n')
        f.write(f'  9. Within_1m: {within_1m_h:>10.2f} %\n')
        f.write(f' 10. Within_3m: {within_3m_h:>10.2f} %\n\n')

        f.write('高程误差 (U 方向, 取绝对值):\n')
        f.write(f'  1. RMS:       {rmse_u:>10.4f} m\n')
        f.write(f'  2. Mean:      {mean_u:>10.4f} m\n')
        f.write(f'  3. Median:    {median_u:>10.4f} m\n')
        f.write(f'  4. Std:       {std_u:>10.4f} m\n')
        f.write(f'  5. P68:       {p68_u:>10.4f} m\n')
        f.write(f'  6. P95:       {p95_u:>10.4f} m\n')
        f.write(f'  7. RMS.68:    {rms68_u:>10.4f} m\n')
        f.write(f'  8. RMS.95:    {rms95_u:>10.4f} m\n')
        f.write(f'  9. Within_1m: {within_1m_u:>10.2f} %\n')
        f.write(f' 10. Within_3m: {within_3m_u:>10.2f} %\n\n')

    f.write('='*80+'\n按 Dataset 分组统计\n'+'='*80+'\n\n')
    f.write(df_dataset_stats.to_string(index=False)+'\n\n')
    f.write('='*100+'\n按 Phone 分组统计\n'+'='*100+'\n\n')
    f.write(df_phone_stats.to_string(index=False)+'\n\n')
    for tag in pivot_rms_h:
        f.write('='*80+f'\nDataset x Phone RMS ({tag} - 水平误差)\n'+'='*80+'\n\n')
        f.write(pivot_rms_h[tag].round(4).to_string()+'\n\n')
    # 添加竖直方向 RMS 热力图结果
    for tag in pivot_rms_u:
        f.write('='*80+f'\nDataset x Phone RMS ({tag} - 竖直方向误差)\n'+'='*80+'\n\n')
        f.write(pivot_rms_u[tag].round(4).to_string()+'\n\n')
    f.write('\n'+'='*80+'\n生成的可视化文件\n'+'='*80+'\n\n')
    for fn, desc in [('01_cdf_*.png','CDF'),('02_heatmap_*.png','热力图'),('03_histogram_*.png','直方图'),('04_boxplot_*.png','箱线图'),('05_bar_*.png','RMS柱状图')]:
        f.write(f'  \u2022 {fn:<40} - {desc}\n')
    f.write('\n'+'='*100+'\n报告生成完成\n'+'='*100+'\n')

print(f'\n\u2713 分析报告已生成: {report_file}')
print(f'\n报告摘要（前60行）:')
with open(report_file, 'r', encoding='utf-8') as rf:
    lines = rf.readlines()
    print(''.join(lines[:60]))